# Data Science Bootcamp – MovieLens Project
## Team Members:
- Svyatoslav / qarlbass
- Bogdan / gentlemk

___

## Оглавление

* [Исследование вселейнной MovieLens](#intro)
* [Описание набора данных](#dataset)
* [Архитектура аналитического пайплайна](#classes)
    * [Часть 1. Анализ данных из movies.csv](#part1)
    * [Часть 2. Анализ аудитории и оценок (класс Ratings)](#part2)
    * [Часть 3. Анализ данных из links.csv](#part3)
    * [Часть 4. Сенантический анализ (класс Tags)](#part4)
* [Общий вывод по проекту MovieLens](#conclusion)


<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->

## <a id='intro'></a>Исследование вселенной MovieLens

### Контекст и цели исследования

Современные стриминговые платформы ежедневно сталкиваются с миллионами выборов, которые делают пользователи. Каким образом алгоритмы понимают, что нам понравится? В основе этих систем лежит анализ исторических данных: оценок, отзывов и метаданных фильмов.

В данном исследовании мы погружаемся во вселенную **MovieLens** — классический датасет для построения рекомендательных систем. Мы пройдем путь от базовых характеристик контента (годы выпуска, жанры, бюджеты) до сложных аспектов поведения аудитории: разберемся, как люди ставят оценки, почему одни фильмы вызывают споры, а другие становятся классикой, и как извлечь смысл из хаотичных текстовых тегов.

Целью данной работы является всесторонний анализ коллекции фильмов и пользовательской активности. Для решения этой задачи мы применяем методы разведочного анализа данных (EDA). Исследование охватывает четыре ключевых измерения:

1. **Структура контента:** анализ временных периодов (распределение по годам) и жанрового разнообразия.
2. **Коммерческие метрики:** обогащение данных информацией о бюджетах, кассовых сборах и режиссерах.
3. **Поведение пользователей:** выявление статистических смещений в оценках и поиск аномалий.
4. **Семантика:** анализ пользовательских тегов как источника качественных признаков для будущих моделей машинного обучения.

___

## <a id='dataset'></a>Описание набора данных



В проекте используется датасет [MovieLens Latest Small](https://grouplens.org/datasets/movielens/latest/), состоящий из четырёх CSV-файлов: `movies.csv`, `ratings.csv`, `tags.csv` и `links.csv`.

Каждый файл содержит отдельную часть информации о фильмах и пользовательской активности.

### movies.csv

Содержит информацию о фильмах:
- `movieId` — уникальный идентификатор фильма;
- `title` — название фильма и год выпуска;
- `genres` — жанры фильма. Если фильм относится к нескольким жанрам, они разделяются символом `|`.

### ratings.csv

Содержит пользовательские оценки фильмов:
- `userId` — идентификатор пользователя;
- `movieId` — идентификатор фильма;
- `rating` — оценка фильма по шкале от 0.5 до 5.0;
- `timestamp` — время выставления оценки в формате Unix timestamp.

### tags.csv

Содержит пользовательские теги, добавленные к фильмам:
- `userId` — идентификатор пользователя;
- `movieId` — идентификатор фильма;
- `tag` — текстовый тег или описание фильма;
- `timestamp` — время добавления тега в формате Unix timestamp.

### links.csv

Содержит идентификаторы фильмов на сторонних платформах:
- `movieId` — идентификатор фильма в MovieLens;
- `imdbId` — идентификатор фильма в IMDb;
- `tmdbId` — идентификатор фильма в TMDB.


___

## <a id='classes'></a>Архитектура аналитического пайплайна

### Проектирование системы классов

Для обработки и анализа данных была спроектирована объектно-ориентированная архитектура. Вместо разрозненных скриптов, исследование разбито на четыре логических модуля, инкапсулированных в едином файле `movielens_analysis.py`:

- `Movies` — работа с основной информацией о фильмах: названиями, годами выпуска и жанрами.
- `Links` — получение и анализ дополнительной информации о фильмах с платформы TMDB: режиссёров, бюджетов, кассовых сборов и продолжительности.
- `Ratings` — анализ пользовательских оценок, активности пользователей и распределения рейтингов.
- `Tags` — анализ пользовательских тегов, добавленных к фильмам.

Подключим классы и создадим их экземпляры, которые будем использовать для выполнения дальнейшего анализа.

In [1]:
from movielens_analysis import Movies, Links, Ratings, Tags

movies = Movies()
links = Links()
ratings = Ratings()
tags = Tags()

### <a id='part1'></a>Часть 1. Анализ данных из movies.csv

#### Распределение фильмов по годам выпуска dist_by_release()

Для начала рассмотрим, как фильмы распределены по годам выпуска. Метод `dist_by_release()` подсчитывает количество фильмов для каждого года и сортирует результаты по убыванию количества.

In [2]:
%timeit -n 1 -r 1 movies.dist_by_release()
release_distribution = movies.dist_by_release()
release_distribution

10.5 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


{2002: 311,
 2006: 295,
 2001: 294,
 2007: 284,
 2000: 283,
 2009: 282,
 2003: 279,
 2004: 279,
 2014: 278,
 1996: 276,
 2015: 274,
 2005: 273,
 2008: 269,
 1999: 263,
 1997: 260,
 1995: 259,
 1998: 258,
 2011: 254,
 2010: 247,
 2013: 239,
 1994: 237,
 2012: 233,
 2016: 218,
 1993: 198,
 1992: 167,
 1988: 165,
 1987: 153,
 1990: 147,
 1991: 147,
 2017: 147,
 1989: 142,
 1986: 139,
 1985: 126,
 1984: 101,
 1981: 92,
 1980: 89,
 1982: 87,
 1983: 83,
 1979: 69,
 1977: 63,
 1973: 59,
 1978: 59,
 1965: 47,
 1971: 47,
 1974: 45,
 1976: 44,
 1964: 43,
 1967: 42,
 1968: 42,
 1975: 42,
 1966: 42,
 2018: 41,
 1962: 40,
 1972: 39,
 1963: 39,
 1959: 37,
 1960: 37,
 1955: 36,
 1969: 35,
 1961: 34,
 1970: 33,
 1957: 33,
 1958: 31,
 1953: 30,
 1956: 30,
 1940: 25,
 1949: 25,
 1954: 23,
 1942: 23,
 1939: 23,
 1946: 23,
 1951: 22,
 1950: 21,
 1947: 20,
 1948: 20,
 1941: 18,
 1936: 18,
 1945: 17,
 1937: 16,
 1952: 16,
 1944: 16,
 1938: 15,
 1931: 14,
 1935: 13,
 1933: 12,
 1934: 11,
 1943: 10,
 1932: 9,

Распределение по годам получилось ровным, без выраженных пиков. Сверху 2002 год — 311 фильмов, за ним 2006 (295), 2001 (294), 2007 (284), 2000 (283). Первая десятка укладывается в 276–311, разброс всего 35 фильмов на десять позиций. То есть верхняя часть списка плоская: нельзя сказать, что какой-то один год доминирует в коллекции.

Основная масса фильмов приходится на 1990-е — 2010-е. В первой десятке семь лет из 2000-х и 2010-х плюс 1996-й. Середину списка занимают остальные годы 1990-х: 1999 (263), 1997 (260), 1995 (259), 1998 (258), 1994 (237), 1993 (198), 1992 (167). Дальше идёт плавный спад.

#### Распределение фильмов по жанрам dist_by_genres()

Для анализа жанрового состава коллекции используется метод `dist_by_genres()`. Он подсчитывает количество фильмов, относящихся к каждому жанру.

In [3]:
%timeit -n 1 -r 1 movies.dist_by_genres()
genre_distribution = movies.dist_by_genres()
genre_distribution

3.26 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


{'Drama': 4361,
 'Comedy': 3756,
 'Thriller': 1894,
 'Action': 1828,
 'Romance': 1596,
 'Adventure': 1263,
 'Crime': 1199,
 'Sci-Fi': 980,
 'Horror': 978,
 'Fantasy': 779,
 'Children': 664,
 'Animation': 611,
 'Mystery': 573,
 'Documentary': 440,
 'War': 382,
 'Musical': 334,
 'Western': 167,
 'IMAX': 158,
 'Film-Noir': 87,
 '(no genres listed)': 34}

Результаты показывают, что наиболее распространённым жанром является Drama — он встречается у 4361 фильмов. На втором месте находится Comedy с 3756 фильмами. Также широко представлены Romance (1539), Thriller (179) и Action (1828).

Наименее распространёнными жанрами оказались IMAX (158), Film-Noir (87) и Western (167).

#### Фильмы с наибольшим количеством жанров most_genres(n)

Метод `most_genres(n)` позволяет определить фильмы с наибольшим количеством указанных жанров. Это помогает оценить жанровое разнообразие отдельных фильмов.

In [4]:
%timeit -n 1 -r 1 movies.most_genres(10)
most_genres = movies.most_genres(10)
most_genres

6.07 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


{'Rubber (2010)': 10,
 'Patlabor: The Movie (Kidô keisatsu patorebâ: The Movie) (1989)': 8,
 'Mulan (1998)': 7,
 'Who Framed Roger Rabbit? (1988)': 7,
 'Osmosis Jones (2001)': 7,
 'Interstate 60 (2002)': 7,
 'Robots (2005)': 7,
 'Pulse (2006)': 7,
 'Aqua Teen Hunger Force Colon Movie Film for Theaters (2007)': 7,
 'Enchanted (2007)': 7}

Среди лидеров много анимации и семейного кино: Mulan, Who Framed Roger Rabbit?, Osmosis Jones, Robots, Enchanted, Tangled, Mars Needs Moms, The Lion King, Beauty and the Beast, All Dogs Go to Heaven 2. У таких фильмов обычно сразу несколько жанровых ярлыков — Animation, Children, Adventure, Comedy, Musical, Family. Разметка щедрая.

Rubber с десятью жанрами — отдельный случай. Это абсурдистская комедия ужасов про ожившую покрышку, и её жанровый набор, судя по всему, отражает именно нежелание разметчиков выбирать: сюда попали и Comedy, и Horror, и Drama, и Mystery, и Thriller, и всё остальное. Десять жанров на один фильм — это уже не описание, а перечисление.

Strange Days и Super Mario Bros. — примеры другого рода: фантастика и боевик, которым навесили ещё и триллер, и драму, и приключения. Тут скорее разметка «на всякий случай», чем реальное жанровое разнообразие.

#### Поиск фильмов по жанру search_by_genre(genre)

После изучения общего распределения жанров рассмотрим возможность выделить фильмы, относящиеся к конкретной категории. Это позволяет подробнее изучить отдельные жанры и получить выборку фильмов для дальнейшего анализа.
Для примера выберем жанр `Drama` и посмотрим, сколько фильмов к нему относится, а также какие картины входят в эту категорию.

In [5]:
%timeit -n 1 -r 1 movies.search_by_genre('Drama')
drama_movies = movies.search_by_genre('Drama')

len(drama_movies), drama_movies[:10]

1.44 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


(4361,
 ['Waiting to Exhale (1995)',
  'American President, The (1995)',
  'Nixon (1995)',
  'Casino (1995)',
  'Sense and Sensibility (1995)',
  'Money Train (1995)',
  'Copycat (1995)',
  'Powder (1995)',
  'Leaving Las Vegas (1995)',
  'Othello (1995)'])

Нашлось 4361 фильмов — ровно столько же, сколько показал dist_by_genres() для Drama. Совпадение ожидаемое, но важное: два независимых метода дают одну и ту же цифру, значит разбор поля genres работает корректно и в агрегированном подсчёте, и в фильтрации по конкретному жанру.

В первой десятке — Waiting to Exhale, American President, Nixon, Casino, Sense and Sensibility, Money Train, Copycat, Powder, Leaving Las Vegas, Othello. Все десять — 1995 года. Не случайность: 1995-й — один из самых плотных годов в выборке (в dist_by_release() у него 259 фильмов, 16-е место).

#### Выводы по анализу movies.csv

По годам картина ровная, без выраженных пиков. Сверху 2002 год — 311 фильмов, за ним 2006 (295), 2001 (294), 2007 (284), 2000 (283). Первая десятка укладывается в 276–311, разброс всего 35 фильмов на десять позиций. Основная масса приходится на 1990-е — 2010-е, а в хвосте распределения — единичные записи 1902–1917 годов.

В жанровом составе два явных лидера — Drama и Comedy. Точные цифры: Drama — 4361 фильм, Comedy — 3756. Вместе они покрывают больше половины выборки. Дальше с заметным отрывом идут Romance, Thriller, Action. В хвосте — Documentary, Western, Film-Noir и IMAX, который жанром в привычном смысле не является. Всего 19 категорий, сумма вхождений — 2 235 при 1 000 фильмов, то есть в среднем около 2,2 жанра на фильм.

Жанры пересекаются, и это хорошо видно на фильмах-рекордсменах. Максимум в выборке — 10 жанров у Rubber (2010). Дальше Patlabor: The Movie с восемью и двенадцать фильмов с семью: Mulan, Who Framed Roger Rabbit?, Osmosis Jones, Interstate 60, Robots, Pulse, Aqua Teen Hunger Force Colon Movie Film for Theaters, Enchanted, Aelita: The Queen of Mars, Inception, Tangled, Mars Needs Moms. Среди лидеров много анимации и семейного кино — у таких фильмов обычно сразу несколько ярлыков. Но «больше жанров» здесь означает лишь то, что фильм сложно отнести к одной категории, а не то, что он содержательно разнообразнее.

Отдельно проверили жанр Drama: поиск вернул 4361 название. Первые десять найденных драм — все 1995 года, что согласуется с общей картиной по годам: 1995-й входит в топ-20 по плотности, и Drama там представлена широко.

---


### <a id='part2'></a>Часть 2. Анализ аудитории и оценок (класс Ratings)

#### Распределение оценок: Портрет зрителя

> Проанализируем то, ради чего (а точнее кого) снимаются фильмы - зрители. Одним людям нравятся фильмы с интересным сюжетом, другие люди терпеть не могут романтичные фильмы. Чтобы оценить свои предпочтения, зрители выставляют оценки. В нашем случае оценка фильма может быть от 0.5 до 5.0 с шагом 0.5. Посмотрим, какие оценки пользователи ставили чаще всего:
>

In [6]:
%timeit -n 1 -r 1 
dist_ratings = ratings.movies_analytics.dist_by_rating()
for rating, count in dist_ratings.items():
    print(f"{rating}: {'#' * (count // 1000 + 1)} ({count})")

0.5: ## (1370)
1.0: ### (2811)
1.5: ## (1791)
2.0: ######## (7551)
2.5: ###### (5550)
3.0: ##################### (20047)
3.5: ############## (13136)
4.0: ########################### (26818)
4.5: ######### (8551)
5.0: ############## (13211)


В данных прослеживается, что распределение рейтингов имеет явно выраженный дискретный характер с модами на целые числа. Главный пик приходится на 4 (26818 записи), далее следуют 3 (20047 записи) и 5 (13211 записей). 
Обратите внимание на ассиметрию: оценок ниже 2.5 значительно меньше, в то время как оценок выше 3.0 — более 80000. Это означает наличие сильного положительного смещения в данных MovieLens. Пользователи оставляют отзывы в основном на те фильмы, которые им понравились, либо склонны завышать баллы. Также заметно, что половинчатые оценки (Х.5) используются неохотно, за исключением 4.5 и 3.5 — видимо, это способ сказать «почти идеально, но есть к чему придраться».

#### Топ по популярности

> Какие фильмы скрываются за этими оценками? Посмотрим топ 10 фильмов по количеству оценок:

In [7]:
%timeit -n 1 -r 1 
top_num_r = ratings.movies_analytics.top_by_num_of_ratings(10)
for title, count in top_num_r.items():
    print(f"{title}: ({count})")

Forrest Gump (1994): (329)
Shawshank Redemption, The (1994): (317)
Pulp Fiction (1994): (307)
Silence of the Lambs, The (1991): (279)
Matrix, The (1999): (278)
Star Wars: Episode IV - A New Hope (1977): (251)
Jurassic Park (1993): (238)
Braveheart (1995): (237)
Terminator 2: Judgment Day (1991): (224)
Schindler's List (1993): (220)


Мы наблюдаем картины, которые известны практически каждому человеку во всем мире, такие как *«Форрест Гамп»*, *«Матрица»*, *«Звёздные войны»* и так далее. Но давайте обратим внимание на то, что практически все они сняты в 90-х. Это не случайность. Скорее всего, мы видим перед собой портрет аудитории, которая сформировала свой вкус в эпоху видеосалонов и DVD. Мы можем сделать вывод о демографическом профиле аудитории: это, скорее всего, миллениалы, которые смотрели эти фильмы в юности и теперь ставят этим фильмам высокие оценки, так как ностальгируют по временам своей молодости. 

#### Временное смещение: Когда ставят оценки?

> Чтобы убедиться в этом, давайте посмотрим, в какое время были выставлены оценки

In [8]:
%timeit -n 1 -r 1 
dist_years = ratings.movies_analytics.dist_by_year()
for year, count in dist_years.items():
    print(f"{year}: {'#' * (count // 1000 + 1)} ({count})")

1996: ####### (6040)
1997: ## (1916)
1998: # (507)
1999: ### (2439)
2000: ########### (10061)
2001: #### (3922)
2002: #### (3478)
2003: ##### (4014)
2004: #### (3279)
2005: ###### (5813)
2006: ##### (4059)
2007: ######## (7114)
2008: ##### (4351)
2009: ##### (4158)
2010: ### (2301)
2011: ## (1690)
2012: ##### (4656)
2013: ## (1664)
2014: ## (1439)
2015: ####### (6616)
2016: ####### (6703)
2017: ######### (8198)
2018: ####### (6418)


Этот график иллюстрирует жизненный цикл платформы и проблему распределения данных во времени:

1. **Пик 2000 года**. Аномальный всплеск (10061 оценка). Возможно, именно тогда платформа переживала свою первую волну популярности, люди массово начинали регистрироваться и активно пользоваться платформой, чтобы оценить фильмы своей юности.
2. **Период спада (2010 - 2014)**: Резкое падение активности. Возможно, это связано с тем, что платформа стала некоторым проектом для исследователей, а не массовым сервисом, каким был Netflix.
3. **Возрождение (2015 - 2018)**: Рост активности перед финальным срезом датасета.

> С точки зрения алгоритмов машинного обучения этот график - важный сигнал: у нас есть временное смещение. Если мы будем обучать модель на этих данных, она будет лучше предсказывать вкусы пользователей 2000-х и 2017-х годов. Однако для периода 2010 - 2014 годов данных в разы меньше. Это значит, что если завтра модель встретит пользователя, чей профиль активности пришелся на 2012 год, ей будет сложнее подобрать ему рекомендации - у нее просто относительно мало исторических данных.

#### Проблема малой выборки и взвешенный рейтинг.

##### Фильмы с высоким средним баллом.

> Мы выяснили, какие фильмы самые популярные. Но популярность не всегда равна качеству. Давайте посмотрим на фильмы с высоким средним баллом.

In [27]:
%timeit -n 1 -r 1
rating_average = ratings.movies_analytics.top_by_ratings(10000000, metric='average')
print(f'Количество фильмов с оценкой 5.0: {len([rat for _, rat in rating_average.items() if rat == 5.0])}')
for movie in list(rating_average.items())[:10]:
    print(movie)

Количество фильмов с оценкой 5.0: 296
('The Jinx: The Life and Deaths of Robert Durst (2015)', 5.0)
('Galaxy of Terror (Quest) (1981)', 5.0)
('Alien Contamination (1980)', 5.0)
("I'm the One That I Want (2000)", 5.0)
('Lesson Faust (1994)', 5.0)
('Assignment, The (1997)', 5.0)
('Mephisto (1981)', 5.0)
('Black Mirror', 5.0)
('Dylan Moran: Monster (2004)', 5.0)
('Bill Hicks: Revelations (1993)', 5.0)


Результаты демонстрируют, что в нашей выборке очень много фильмов с оценкой 5.0. Значит ли это, что это самые лучшие фильмы? Однозначно нет. Дело в том, что средний показатель не является самым эффективным для анализа качества фильма, так как фильмы с малым количеством оценок могут иметь 1 или 2 оценки пять, и выбиваться в топ фильмов. Это яркая иллюстрация **проблемы малой выборки**.

Хорошие картины, которые оценили несколько сотен людей с оценкой 4.9, окажутся вне нашего поля зрения. Это несправедливо, а значит необходимо изучить **взвешенный рейтинг** фильмов, чтобы количество проставленных оценок влияло на показатель. Чем больше уровень *поддержки* — тем точнее будет оценен фильм.

##### Отсеиваем шум: Байесовское сглаживание.

> Для этого я создал метод получения взвешенной средней оценки, которая учитывает количество проставленных оценок, а также отсеивает фильмы с минимальным количеством оценок

In [28]:
%timeit -n 1 -r 1
weighted_ratings = ratings.movies_analytics.top_by_weighted_rating(20, min_count_ratings=20)
for title, score in weighted_ratings.items():
    print(f'{title}: {score}')

Shawshank Redemption, The (1994): 4.37
Fight Club (1999): 4.21
Godfather, The (1972): 4.21
Star Wars: Episode IV - A New Hope (1977): 4.18
Usual Suspects, The (1995): 4.17
Godfather: Part II, The (1974): 4.16
Schindler's List (1993): 4.16
Dark Knight, The (2008): 4.15
Goodfellas (1990): 4.15
Matrix, The (1999): 4.15
Pulp Fiction (1994): 4.15
Star Wars: Episode V - The Empire Strikes Back (1980): 4.15
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964): 4.14
Princess Bride, The (1987): 4.14
Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981): 4.14
Departed, The (2006): 4.13
Forrest Gump (1994): 4.13
American History X (1998): 4.12
Casablanca (1942): 4.12
Rear Window (1954): 4.12


Данный список существенно отличается от предыдущих выборок. Применив метод **взвешенного рейтинга**, мы исключили *статистические аномалии* — фильмы с минимальным числом оценок, чей высокий средний балл был обусловлен малой выборкой. Теперь перед нами объективные лидеры кинопроката.

На что указывает этот результат с точки зрения анализа данных?

1. **Высокий уровень поддержки:** В топ попали только фильмы с количеством оценок не менее 20. Их рейтинг — это не случайное совпадение, а результат консенсуса широкой аудитории. Присутствие картин вроде *«Побега из Шоушенка»* и *«Крестного отца»* подтверждает, что они сохраняют высокое качество на протяжении десятилетий.

2. **Эффективность сглаживания:** Ни один фильм в списке не имеет идеальной оценки 5.0. Алгоритм скорректировал экстремальные значения к реальности. Это более честный результат: даже у признанных шедевров есть аудитория, которой они не подошли.

3. **Временное смещение:** Большинство фильмов в топе вышли до 2000-х годов. Классические картины успели накопить значительный объем оценок, который удерживает их средний балл на высоком уровне. Современным фильмам потребуется время, чтобы достичь аналогичного уровня поддержки.


##### Вывод по взвешенному рейтингу.

> *Взвешенный рейтинг* показал, что **"лучший фильм"** - это не просто математическая случайность. Это симбиоз высокой оценки и широкого признания аудитории. Именно такой подход используют крупнейшие платформы для построения своих рекомендаций. 

#### Взгляд через медиану: Устойчивость к выбросам.

##### Фильмы с высоким медианным значением.

> Взвешенный рейтинг помог нам найти фильмы с наивысшим средним баллом, подтвержденным сотнями людей. Но среднее арифметическое коварно: если у фильма 100 оценок 5.0 и 20 оценок 1.0 от ярых хейтеров, среднее может сильно просесть.
>
> Чтобы найти фильмы, которые нравятся всем, можем взглянуть на медиану - оценку, которая находится ровно посередине сортированного списка оценок.

In [11]:
%timeit -n 1 -r 1
top_median = ratings.movies_analytics.top_by_ratings(1000000, metric='median')
print(len([r for _, r in top_median.items() if r == 5.0]))
for movie in list(top_median.items())[:10]:
    print(movie)

319
('The Jinx: The Life and Deaths of Robert Durst (2015)', 5.0)
('Galaxy of Terror (Quest) (1981)', 5.0)
('Alien Contamination (1980)', 5.0)
('Troll 2 (1990)', 5.0)
("I'm the One That I Want (2000)", 5.0)
('Chorus Line, A (1985)', 5.0)
("Guess Who's Coming to Dinner (1967)", 5.0)
('Children of the Corn IV: The Gathering (1996)', 5.0)
('Band of Brothers (2001)', 5.0)
('Lesson Faust (1994)', 5.0)


##### Проблема малой выборки.

Как мы видим, метод вернул 319 фильмов с медианой 5.0. Однако их названия — *«Troll 2»*, *«Alien Contamination»*, *«Children of the Corn IV»* — явно не относятся к шедеврам мирового кинематографа.

Почему так произошло? Причина кроется в **малом объеме данных**. Если фильм оценили всего два человека и оба поставили «5», медиана становится 5.0. Алгоритм математически корректен, но результат нерепрезентативен: он не отражает мнение широкой аудитории.

Чтобы найти фильмы, которые действительно нравятся зрителям, необходимо применить **пороговую фильтрацию**. Оставим в базе только те картины, которые собрали минимум 50 оценок. Это обеспечит достаточный уровень доверия к данным, и медиана покажет нам истинных фаворитов публики.

In [12]:
%timeit -n 1 -r 1
top_median_filtered = ratings.movies_analytics.top_by_weighted_rating(20, min_count_ratings=50, metric='median')

for title, score in top_median_filtered.items():
    print(f"{title}: {score}")

Apocalypse Now (1979): 4.5
Casablanca (1942): 4.5
Chinatown (1974): 4.5
Dark Knight, The (2008): 4.5
Departed, The (2006): 4.5
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964): 4.5
Fight Club (1999): 4.5
Godfather, The (1972): 4.5
Godfather: Part II, The (1974): 4.5
Life Is Beautiful (La Vita è bella) (1997): 4.5
Lord of the Rings: The Fellowship of the Ring, The (2001): 4.5
Matrix, The (1999): 4.5
Memento (2000): 4.5
Princess Bride, The (1987): 4.5
Pulp Fiction (1994): 4.5
Rear Window (1954): 4.5
Schindler's List (1993): 4.5
Shawshank Redemption, The (1994): 4.5
Shutter Island (2010): 4.5
Snatch (2000): 4.5


##### Медиана с фильтрацией: Настоящие "народные" хиты.

Вот теперь картина совершенно иная! Отсеяв фильмы, которые смотрели единицы, мы получили список картин с медианой 4.5, подтвержденной десятками зрителей.

**Анализ**:
В отличие от среднего арифметического, медиана показывает нам фильмы с *«нулевой толерантностью к негативу»*. Если у фильма медиана 4.5 при 50+ оценках, это значит, что как минимум половина зрителей поставила ему высший балл, а остальные — очень близкий к нему. В этом списке мы видим уже встречавшиеся нам *"Крестный отец"*, *"Побег из Шоушенка"* и *"Бойцовский клуб"*.

> Для рекомендательных алгоритмов такие фильмы — это «золотой стандарт». Мы можем смело рекомендовать их даже новым пользователям, потому что они вызывают положительные эмоции практически у всех. Следующий шаг — посмотреть на обратную сторону медали: на фильмы, которые наоборот, раскалывают общество на две половины. Перейдем к методу `top_controversial`.

#### Поляризация: Бимодальное распределение

##### Анализ поляризации мнений: Самые спорные фильмы

> До этого мы искали фильмы, которые объединяют людей. Однако кинематограф часто вызывает споры: есть картины, которые одни считают абсолютными шедеврами, а другие — полным провалом. Чтобы найти такие фильмы, мы используем метод `top_controversial`, который вычисляет **дисперсию** оценок.

In [13]:
%timeit -n 1 -r 1
controversial = ratings.movies_analytics.top_controversial(20)

for title, variance in controversial.items():
    print(f"{title}: дисперсия {variance}")

Ivan's Childhood (a.k.a. My Name is Ivan) (Ivanovo detstvo) (1962): дисперсия 5.06
Fanny and Alexander (Fanny och Alexander) (1982): дисперсия 5.06
Troll 2 (1990): дисперсия 4.5
Lassie (1994): дисперсия 4.0
Zed & Two Noughts, A (1985): дисперсия 4.0
Kwaidan (Kaidan) (1964): дисперсия 4.0
Emma (2009): дисперсия 4.0
Play Time (a.k.a. Playtime) (1967): дисперсия 3.72
Room, The (2003): дисперсия 3.56
Peeping Tom (1960): дисперсия 3.5
Opera (1987): дисперсия 3.5
Immortals (2011): дисперсия 3.39
Skin I Live In, The (La piel que habito) (2011): дисперсия 3.38
Host, The (2013): дисперсия 3.38
Phantasm II (1988): дисперсия 3.17
Holiday (1938): дисперсия 3.17
Exorcist III, The (1990): дисперсия 3.12
Clonus Horror, The (1979): дисперсия 3.06
The Purge: Election Year (2016): дисперсия 3.06
To Grandmother's House We Go (1992): дисперсия 3.06


**Дисперсия** — это мера разброса значений относительно среднего. Если фильм получает оценки только 5.0 и 1.0 (игнорируя нейтральные 3.0), его дисперсия достигает математического максимума (5.06 для нашей шкалы). Это маркер *поляризации*: аудитория разделилась ровно 50 на 50. В топе спорных фильмов соседствуют шедевры артхауса *(Иван Грозный, Фанни и Александр)* и нишевый контент *(Troll 2)*.

Мы наблюдаем следующие закономерности:

1. **Ограничение шкалы**: Дисперсия 5.06 — это математический максимум для шкалы от 0.5 до 5.0. Это значит, что аудитория раскололась ровно 50/50.

2. **Сложность для алгоритмов**: Рекомендательные системы ненавидят такие фильмы. Если система порекомендует *«Ивана Грозного»* случайному пользователю, она либо получит восторженный отзыв, либо полностью разрушит доверие пользователя к сервису.

3. **Искажение из-за малой выборки**: В нижней части списка (например, *«To Grandmother's House We Go»*) высокая дисперсия может быть вызвана не гениальностью фильма, а малой выборкой. Если фильм посмотрело 5 человек и двое поставили 1.0, а трое — 5.0, дисперсия взлетит до небес, хотя глобально фильм никому не интересен.

##### Отсев аномалий.

> Однако на этом этапе возникает ловушка: высокая дисперсия может быть вызвана не общественным расколом, а недостатком данных. Если фильм оценили всего двое (один поставил 1.0, другой — 5.0), дисперсия будет максимальной, хотя глобально фильм никому не интересен. В отличие от среднего рейтинга, дисперсию нельзя «сгладить» к общему среднему.
> 
> Чтобы отсеять *статистические аномалии*, мы применим **пороговую фильтрацию**: оставим только те фильмы, которые собрали не менее 50 оценок. Это позволит сфокусироваться на картинах, вызывающих реальный раскол мнений среди широкой аудитории.

In [14]:
%timeit -n 1 -r 1
controversial = ratings.movies_analytics.top_by_weighted_rating(20, min_count_ratings=50, metric='variance')

for title, variance in controversial.items():
    print(f"{title}: дисперсия {variance}")

Blair Witch Project, The (1999): дисперсия 1.86
First Knight (1995): дисперсия 1.66
Austin Powers: The Spy Who Shagged Me (1999): дисперсия 1.46
Moulin Rouge (2001): дисперсия 1.41
Scream (1996): дисперсия 1.41
Mars Attacks! (1996): дисперсия 1.38
Cable Guy, The (1996): дисперсия 1.36
Nutty Professor, The (1996): дисперсия 1.36
Scary Movie (2000): дисперсия 1.34
Rushmore (1998): дисперсия 1.31
Matrix Revolutions, The (2003): дисперсия 1.29
Mummy, The (1999): дисперсия 1.29
Sense and Sensibility (1995): дисперсия 1.29
Big Lebowski, The (1998): дисперсия 1.28
Borat: Cultural Learnings of America for Make Benefit Glorious Nation of Kazakhstan (2006): дисперсия 1.28
Star Wars: Episode II - Attack of the Clones (2002): дисперсия 1.27
Dumb & Dumber (Dumb and Dumber) (1994): дисперсия 1.25
Grease (1978): дисперсия 1.25
Natural Born Killers (1994): дисперсия 1.25
Space Jam (1996): дисперсия 1.25


Вот теперь мы видим истинную картину. После отсечения фильмов с малым количеством оценок, дисперсия снизилась с 5.06 до 1.86, но этот результат гораздо ценнее.

Что мы видим в этом списке?

1. **Культовое кино и «вирусные» релизы**. На первом месте — *«Ведьма из Блэр» (1999)*. Это фильм, который произвел революцию в маркетинге, но многих зрителей укачало от трясущейся камеры и отсутствия монстра. Дисперсия 1.86 означает, что мнения разделились практически поровну: кто-то ставил 5.0 за инновационность, а кто-то — 0.5 за то, что фильм не оправдал ожиданий.

2. **Жанровые крайности и пародии**. В списке много фильмов, которые пародируют другие жанры (*«Остин Пауэрс»*, *«Очень страшное кино»*, *«Марс атакует!»*, *«Борат»*). Юмор — вещь субъективная. Для одного зрителя это гениальная сатира, для другого — нелепый фарс.

3. **Блокбастеры с высокой планкой ожиданий**. *«Матрица: Революция»* (1.29) и *«Звездные войны: Эпизод 2»* (1.27) попали сюда, потому что фанаты ждали от них слишком многого. Высокие ожидания часто приводят к разочарованию (или, наоборот, к фанатичной защите фильма), что и рождает полярные оценки.


Такие фильмы представляют собой **высокорисковые рекомендации**. Алгоритм не может просто сказать *«это хороший фильм»*. Если система порекомендует *«Бората»* случайному пользователю, она с вероятностью 50% угадает вкус, а с вероятностью 50% — вызовет отторжение. Чтобы минимизировать этот риск, системе нужно точно знать вкусовой профиль пользователя и его чувство юмора, а не просто усреднять оценки.

> Итак, мы выяснили, что фильмы бывают массовыми, гениальными и крайне спорными. Но за каждой из этих оценок стоит человек. Некоторые люди ставят пятерки всему подряд, а некоторые — прирожденные критики. Теперь давайте перевернем матрицу и посмотрим на зрителей. Перейдем к классу `Users`


#### Анализ аудитории.

> До этого мы смотрели на фильмы: какие они, как их оценивают и кто в них снимается. Но за каждой оценкой стоит конкретный человек. В рекомендательных системах анализ пользователей не менее важен, чем анализ контента. От того, насколько активен пользователь и как сильно он разбрасывается оценками, зависит, сможем ли мы построить для него качественные персональные рекомендации.
> 
> В этой части мы посмотрим на портрет зрителя с трех сторон: активность, строгость и непоследовательность.


##### Активность пользователей.

Давайте посмотрим, как распределены оценки между пользователями. В любом сообществе есть «молчуны», которые поставили пару оценок и ушли, и «сверхактивные» фанаты, которые оценивают всё подряд. Метод `dist_by_num_of_ratings` покажет нам эту картину.

In [15]:
%timeit -n 1 -r 1
dist_users = ratings.users.dist_by_num_of_ratings()

print("Топ 20 пользователей с самым большим количеством оценок")
print(f"{'UserID':<10} | {'Кол-во оценок':<15}")
print("-" * 30)
for user_id, count in list(dist_users.items())[-20:][::-1]:
    print(f"{user_id:<10} | {count:<15}")
    


Топ 20 пользователей с самым большим количеством оценок
UserID     | Кол-во оценок  
------------------------------
414        | 2698           
599        | 2478           
474        | 2108           
448        | 1864           
274        | 1346           
610        | 1302           
68         | 1260           
380        | 1218           
606        | 1115           
288        | 1055           
249        | 1046           
387        | 1027           
182        | 977            
307        | 975            
603        | 943            
298        | 939            
177        | 904            
318        | 879            
232        | 862            
480        | 836            


##### Анализ: Закон Парето и проблема "Длинного хвоста"

Взгляните на этот топ-20. Пользователь c *ID* 414 поставил почти 2700 оценок. Это в десятки раз больше, чем у обычного зрителя. Мы наблидаем **скошенное распределение**.

Здесь в действии знаменитый *Закон Парето (правило 80/20)*. Крошечный процент сверхактивных пользователей генерирует львиную долю всех данных в датасете. Обратите внимание, что даже в этом топе разброс огромен: если пользователь 480 поставил 826 оценок, то 414 - 2698.

Какой вывод можем сделать?

1. **Смещение данных**: Модель, обученная на этом датасете, будет в первую очередь подстраиваться под вкусы этих нескольких *«сверхактивных»* пользователей. Их поведение станет *«золотым стандартом»* для алгоритма.

2. **Разреженность**: У большинства обычных людей в базе (которых нет в этом топе) всего по 20-50 оценок. Для алгоритма они выглядят как *«белые пятна»*. Рекомендательной системе будет очень сложно найти для них похожих пользователей (соседей), так как данных о них слишком мало.

Именно поэтому крупные стриминговые платформы так борются за активность аудитории. Чем больше оценок ставит каждый пользователь, тем точнее работают алгоритмы.

> Подойдем к проблеме с другой стороны. Посмотрим пользователей, которые стаивли минимум оценок

In [16]:
%timeit -n 1 -r 1

print("Топ 20 пользователей с самым меньшим количеством оценок")
print(f"{'UserID':<10} | {'Кол-во оценок':<15}")
print("-" * 30)
for user_id, count in list(dist_users.items())[:20]:
    print(f"{user_id:<10} | {count:<15}")

Топ 20 пользователей с самым меньшим количеством оценок
UserID     | Кол-во оценок  
------------------------------
53         | 20             
147        | 20             
189        | 20             
194        | 20             
207        | 20             
257        | 20             
278        | 20             
320        | 20             
406        | 20             
431        | 20             
442        | 20             
569        | 20             
576        | 20             
595        | 20             
26         | 21             
37         | 21             
49         | 21             
87         | 21             
157        | 21             
245        | 21             


Интересная особенность этого датасета: даже самые пассивные пользователи поставили минимум 20 оценок. Это говорит о высоком пороге входа — люди регистрировались на платформе с серьезным намерением оценить большое количество фильмов, либо в наш датасет добавляли пользователей, которые проставили 20 или более оценок.

##### Анализ концентрации данных.

> Теперь изучим какой процент оценок проставили 20 пользователей с наибольшим количеством оценок

In [17]:
%timeit -n 1 -r 1
total_rating = sum(dist_users.values())
total_top_20_rating = sum(list(dist_users.values())[-20:])
print('Процент количества оценок 20 активных пользователей от общего количества:')
print(f'{round(total_top_20_rating / total_rating * 100, 2)}%')

Процент количества оценок 20 активных пользователей от общего количества:
25.62%


Полученный результат **25.62%** показывает, что всего 20 пользователель сгенерировали более четверти всех оценок в базе. Это яркий пример **смещения выборки**. Наш датасет отражает не столько мнение широкой аудитории, сколько предпочтения небольшой группы сверхактивных зрителей.

**Влияние на рекомендательные системы**:

1. **Смещение рекомендаций**: Алгоритм, обученный на таких данных, будет склонен переоценивать фильмы, популярные среди этих 20 пользователей, создавая *«эффект эха»*.

2. **Проблема холодного старта**: Для пользователей, поставивших 10–20 оценок, у модели будет недостаточно исторических данных, чтобы найти качественные закономерности и построить персонализированные рекомендации.

Для борьбы с этим в индустрии применяется нормализация весов. Вклад сверхактивного пользователя с 2000 оценок не должен доминировать над вкладом сотни обычных зрителей, иначе модель теряет объективность.

##### Строгость пользователей: Кто ставит низкие оценки?

> Мы выяснили, что небольшая группа людей генерирует огромную часть оценок. Но одинаково ли они оценивают кино? Все мы знаем типажи: кому-то нравится абсолютно всё (их средняя оценка близка к 5.0), а кто-то — прирожденный критик, который редко ставит выше тройки. 
>
> Давайте используем метод `dist_by_ratings` с метрикой `average`, чтобы найти самых лояльных и самых строгих зрителей в нашей базе.

In [22]:
%timeit -n 1 -r 1
user_means = ratings.users.dist_by_ratings(metric='average')
user_counts = ratings.users.dist_by_num_of_ratings()

filtered_means = {
    uid: mean for uid, mean in user_means.items()
    if user_counts.get(uid, 0) >= 50
}
# Сортируем, чтобы найти крайности
sorted_means = sorted(filtered_means.items(), key=lambda x: x[1])

print("Самые строгие критики (низкий средний балл):")
for uid, mean in sorted_means[:5]:
    print(f"  Пользователь {uid:<5} | Средняя оценка: {mean}")

print("\nСамые лояльные зрители (высокий средний балл):")
# Выводим в обратном порядке, чтобы самые высокие были первыми
for uid, mean in sorted_means[-5:][::-1]: 
    print(f"  Пользователь {uid:<5} | Средняя оценка: {mean}")

Самые строгие критики (низкий средний балл):
  Пользователь 139   | Средняя оценка: 2.14
  Пользователь 153   | Средняя оценка: 2.22
  Пользователь 567   | Средняя оценка: 2.25
  Пользователь 298   | Средняя оценка: 2.36
  Пользователь 517   | Средняя оценка: 2.39

Самые лояльные зрители (высокий средний балл):
  Пользователь 523   | Средняя оценка: 4.69
  Пользователь 348   | Средняя оценка: 4.67
  Пользователь 171   | Средняя оценка: 4.63
  Пользователь 452   | Средняя оценка: 4.56
  Пользователь 122   | Средняя оценка: 4.55


##### Субъективность оценок

Применив фильтр по количеству оценок (не менее 50), мы избавились от статистического шума, когда пользователь с двумя оценками *«пятерками»* объявлялся самым лояльным зрителем. Теперь мы видим истинный портрет аудитории.

Мы наблюдаем ярко выраженное **смещение оценщика**. Разброс средних оценок очень велик: от 2.14 до 4.69.

* Строгие критики *(средний балл ~2.1–2.3)*: Эти пользователи посмотрели десятки фильмов, но большинство из них им не понравились. Они ставят оценки очень придирчиво.

* Лояльные зрители *(средний балл ~4.6)*: Эти люди склонны хвалить кинематограф. Даже если фильм объективно средний, они, скорее всего, поставят ему 4 или 5.

> Почему это важно для рекомендательных систем?
Представьте, что алгоритм просто усреднит оценки фильма. Если фильм посмотрели только *«строгие критики»*, его средний балл будет 2.5, и система посчитает его плохим. Но если бы этот же фильм посмотрели *«лояльные зрители»*, он получил бы 4.5.
> 
> Чтобы избежать этого искажения, в алгоритмах фильтрации применяется нормализация оценок. Из каждой оценки пользователя вычитается его собственный средний балл. Это позволяет привести строгих критиков и добряков к единой шкале и объективно сравнить их вкусы.

##### Самые противоречивые пользователи

> В разделе про фильмы мы искали картины, которые раскололи общество (высокая дисперсия оценок). Но что насчет самих зрителей? Кто из них ставит оценки «от балды», меняя 5.0 на 0.5? 
>
> Метод `top_controversial` в классе `Users` вычисляет дисперсию оценок для каждого пользователя. Чем выше дисперсия — тем менее последователен зритель в своих предпочтениях. Чтобы избежать влияния статистического шума (когда у пользователя всего 2-3 оценки), мы применим фильтр: оставим только тех, кто поставил не менее 50 оценок.

In [3]:
%timeit -n 1 -r 1
controversial_users_raw = ratings.users.top_controversial(10000)
user_counts = ratings.users.dist_by_num_of_ratings()

filtered_controversial = {
    uid: var for uid, var in controversial_users_raw.items()
    if user_counts.get(uid, 0) >= 50
}

top_10_controversial = dict(list(filtered_controversial.items())[:20])

print(f"{'UserID':<10} | {'Дисперсия':<15}")
print("-" * 30)
for uid, var in top_10_controversial.items():
    print(f"{uid:<10} | {var:<15}")

UserID     | Дисперсия      
------------------------------
393        | 2.61           
160        | 2.39           
22         | 2.36           
418        | 2.25           
308        | 2.23           
287        | 2.21           
112        | 2.12           
153        | 2.04           
191        | 2.03           
266        | 2.03           
416        | 2.01           
310        | 1.9            
594        | 1.84           
34         | 1.82           
373        | 1.81           
517        | 1.81           
542        | 1.78           
500        | 1.77           
7          | 1.76           
143        | 1.74           


##### Анализ: Высокая дисперсия как индикатор сложного профиля

После отсечения пользователей с малым количеством оценок (менее 50), картина стала реалистичнее. Мы видим пользователей с дисперсией от 2.61 до 1.74. Такой разброс означает, что они ставят как максимальные, так и минимальные баллы при достаточном объеме просмотров. Таких пользователей можно отнести к категории *сложных профилей*.

Почему это создает трудности для рекомендательных систем?

1. **Высокая ошибка предсказания**: Рекомендательным системам крайне сложно предсказать оценки таких людей. Их предпочтения не описываются простыми линейными закономерностями, так как они могут высоко оценить сложный артхаус и низко — популярный блокбастер.

2. **Нишевые интересы**: Высокая дисперсия часто указывает на то, что человек смотрит фильмы из совершенно разных, иногда противоположных жанровых категорий. Алгоритму сложно найти для него «соседей» с похожим набором вкусов.

3. **Проблема *"Шума"* в профиле**: Оценки таких пользователей могут существенно искажать средние показатели фильмов. Без применения нормализации оценок
 их полярное мнение будет доминировать и снижать объективность рекомендаций.



#### Итоговые выводы по части 2 (Класс Movies и Users):

Проведя глубокий анализ оценок, мы выявили несколько фундаментальных закономерностей, которые критически важны для понимания работы рекомендательных систем:

1. **Положительное смещение:** Пользователи склонны ставить высокие оценки. Большая часть рейтингов приходится на 3.0, 4.0 и 5.0, а низкие баллы используются крайне редко. Это означает, что алгоритмам нужно быть осторожными: средняя оценка фильма в 4.0 может означать как *"шедевр"*, так и *"проходное кино"*.

2. **Проблема малой выборки и взвешенный рейтинг:** Обычное среднее арифметическое не работает, так как фильмы с 1-2 случайными пятерками выбиваются в топ. Мы решили это с помощью **байесовского сглаживания**, которое учитывает уровень поддержки. Это позволило выявить объективную классику.

3. **Смещение данных и Закон Парето:** Аудитория крайне неоднородна. Крошечный процент сверхактивных пользователей генерирует четверть всех оценок. Датасет смещен в сторону их вкусов.

4. **Субъективность оценок:** Зрители оценивают кино по-разному. Есть *"добряки"* со средним баллом 4.6, и *"критики"* со средним баллом 2.1. Для алгоритмов это означает, что оценки разных людей нельзя просто складывать — требуется **нормализация**.

**Общий итог:** Мы доказали, что сырые данные в рекомендательных системах практически бесполезны. Чтобы найти настоящие шедевры и понять зрителя, необходимо применять статистическую фильтрацию, сглаживание и нормализацию. Именно поэтому построение качественного рекомендательного движка — это сложная инженерная и Data Science задача.

---

### <a id='part3'></a>Часть 3. Анализ данных из links.csv

Класс `Links` позволяет дополнить информацию из датасета сведениями о фильмах, полученными с платформы TMDB. Для этого используются идентификаторы фильмов, связывающие записи с соответствующими страницами TMDB.

Полученные данные позволяют изучить не только характеристики самих фильмов, но и особенности их производства и коммерческие показатели: режиссёров, бюджеты, кассовые сборы и продолжительность.

На основе этих данных можно определить режиссёров с наибольшим количеством фильмов в выборке, сравнить бюджеты и сборы картин, а также изучить продолжительность фильмов.

#### Получение дополнительной информации о фильмах `get_imdb()` и `select_movies`

Для проведения анализа необходимо дополнить данные MovieLens информацией о фильмах с платформы TMDB.

Для формирования выборки используется метод `select_movies(start_id, end_id)`. Он позволяет выбрать фильмы по диапазону идентификаторов MovieLens.

После выбора полей метод `get_imdb()` получает со страниц TMDB дополнительную информацию:
- название фильма;
- режиссёра;
- бюджет;
- мировые кассовые сборы;
- продолжительность.

Для анализа взяли фильмы с идентификаторами от 100 до 250. В результате `select_movies()` вернул 127 идентификаторов, для которых далее выполнялась загрузка дополнительной информации.

Полнота полученных данных зависит от информации, доступной на TMDB: у некоторых фильмов могут отсутствовать сведения о бюджете, кассовых сборах или режиссёре.

In [2]:
%timeit -n 1 -r 1 links.select_movies(100, 250)
movie_ids = links.select_movies(100, 250)
len(movie_ids), movie_ids[:10]

321 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


(127, [100, 101, 102, 103, 104, 105, 106, 107, 108, 110])

В итоге по диапазону ID от 100 до 250 было отобрано 127 фильмов. Полученный список будет использоваться для загрузки дополнительной информации с TMDB.

In [3]:
%timeit -n 1 -r 1 
links.get_imdb(
movie_ids,["Title", "Runtime", "Budget", "Cumulative Worldwide Gross", "Director"])
links.imdb_info

[[250,
  'Heavyweights (Heavy Weights) (1995)',
  '1h 37m',
  'Budget $10,000,000',
  'Revenue $17,700,000',
  'Steven Brill'],
 [249,
  'Immortal Beloved (1994)',
  '2h 1m',
  'Budget $120',
  'Revenue -',
  'Bernard Rose'],
 [248,
  'Houseguest (1994)',
  '1h 49m',
  'Budget $10,500,000',
  'Revenue $26,300,000',
  'Randall Miller'],
 [247,
  'Heavenly Creatures (1994)',
  '1h 49m',
  'Budget $5,000,000',
  'Revenue $3,049,135',
  'Peter Jackson'],
 [246,
  'Hoop Dreams (1994)',
  '2h 54m',
  'Budget $700,000',
  'Revenue $7,830,611',
  'Steve James'],
 [243,
  'Gordy (1995)',
  '1h 30m',
  'Budget -',
  'Revenue $3,992,809',
  'Mark Lewis'],
 [242,
  'Farinelli: il castrato (1994)',
  '1h 51m',
  'Budget -',
  'Revenue -',
  'Gérard Corbiau'],
 [241,
  'Fluke (1995)',
  '1h 36m',
  'Budget $15,000,000',
  'Revenue $4,000,000',
  'Carlo Carlei'],
 [240,
  'Hideaway (1995)',
  '1h 43m',
  'Budget $15,000,000',
  'Revenue $12,201,255',
  'Brett Leonard'],
 [239,
  'Goofy Movie, A (1995

По результатам загрузки удалось получить дополнительную информацию о фильмах из TMDB: названия, имена режиссёров, бюджеты, мировые кассовые сборы и продолжительность.

#### Режиссёры с наибольшим количеством фильмов `top_directors(n)`

Метод `top_directors(n)` подсчитывает количество фильмов каждого режиссёра в загруженной выборке и возвращает список наиболее часто встречающихся режиссёров.

In [4]:
%timeit -n 1 -r 1 links.top_directors(20)
links.top_directors(20)

176 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


{'Kevin Smith': 2,
 'Steven Brill': 1,
 'Bernard Rose': 1,
 'Randall Miller': 1,
 'Peter Jackson': 1,
 'Steve James': 1,
 'Mark Lewis': 1,
 'Gérard Corbiau': 1,
 'Carlo Carlei': 1,
 'Brett Leonard': 1,
 'Kevin Lima': 1,
 'Phillip Borsos': 1,
 'Billy Crystal': 1,
 'Lawrence Kasdan': 1,
 'Tim Burton': 1,
 'Garry Marshall': 1,
 'Atom Egoyan': 1,
 'Ang Lee': 1,
 'Peter Farrelly': 1,
 'Taylor Hackford': 1}

Результат короткий: два фильма — только у Kevin Smith (Clerks, Mallrats). Остальные режиссёры представлены по одной картине.

Наша выборка уже позволяет обнаружить повторяющихся режиссёров, однако их количество пока невелико. Для более содержательного сравнения необходимо увеличить объём данных или рассмотреть более широкий диапазон идентификаторов.

#### Фильмы с наибольшим бюджетом `most_expensive(n)`

Рассмотрим распределение бюджетов производства фильмов. Метод `most_expensive(n)` позволяет определить картины с наибольшими затратами на производство и сравнить их финансовые показатели.

Для анализа будут выбраны 20 фильмов с самыми высокими бюджетами среди загруженных данных.

In [5]:
%timeit -n 1 -r 1 links.most_expensive(20)
links.most_expensive(20)

239 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


{'Waterworld (1995)': 175000000,
 'Batman Forever (1995)': 100000000,
 'Judge Dredd (1995)': 90000000,
 'Die Hard: With a Vengeance (1995)': 90000000,
 'Braveheart (1995)': 72000000,
 'Under Siege 2: Dark Territory (1995)': 60000000,
 'Up Close and Personal (1996)': 60000000,
 'Disclosure (1994)': 55000000,
 'First Knight (1995)': 55000000,
 'Crimson Tide (1995)': 53000000,
 'Apollo 13 (1995)': 52000000,
 'Scarlet Letter, The (1995)': 50000000,
 'Congo (1995)': 50000000,
 'Casper (1995)': 50000000,
 'Jade (1995)': 50000000,
 'Drop Zone (1994)': 45000000,
 'Showgirls (1995)': 45000000,
 'Strange Days (1995)': 42000000,
 'Boomerang (1992)': 40000000,
 'City Hall (1996)': 40000000}

Наибольший бюджет среди рассмотренных фильмов Waterworld выбивается сильно, а именно 175 млн, что почти в два раза больше Batman Forever с бюджетом 100 млн, который находится на втором месте. Дальше идёт плотная группа в районе $50–60M: Under Siege 2, Up Close & Personal, Disclosure, First Knight, Crimson Tide. Apollo 13 с $52M выглядит скромно на фоне соседей, хотя по сборам он лидер, а также сразу четыре фильма с ровно $50M (The Scarlet Letter, Congo, Casper, Jade).

#### Фильмы с наибольшей разницей между сборами и бюджетом `most_profitable(n)`

Метод `most_profitable(n)` рассчитывает разницу между кассовыми сборами и бюджетом каждого фильма, после чего сортирует результаты по убыванию. Это выделяет фильмы с наибольшей разницей между полученными сборами и затратами на производство.

In [6]:
%timeit -n 1 -r 1 links.most_profitable(20)
links.most_profitable(20)

216 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


{'Apollo 13 (1995)': 303237933,
 'Die Hard: With a Vengeance (1995)': 276101666,
 'Casper (1995)': 237928194,
 'Batman Forever (1995)': 236529144,
 'Dumb & Dumber (Dumb and Dumber) (1994)': 231275374,
 'Disclosure (1994)': 159015089,
 'Bridges of Madison County, The (1995)': 158000000,
 'Birdcage, The (1996)': 154300000,
 'Braveheart (1995)': 141216216,
 'Bad Boys (1995)': 122407024,
 'Crimson Tide (1995)': 104387195,
 'Nine Months (1995)': 103500000,
 'Congo (1995)': 102022101,
 'Boomerang (1992)': 91052444,
 'Waterworld (1995)': 89218220,
 'Net, The (1995)': 88627965,
 'Species (1995)': 78400000,
 'First Knight (1995)': 72600435,
 'Rumble in the Bronx (Hont faan kui) (1995)': 68528558,
 'Mighty Morphin Power Rangers: The Movie (1995)': 51433194}

Наибольшая разница наблюдается у фильма Apollo 13 — около 303,2 млн. Следом идут Die Hard: With a Vengeance (276,1 млн), Casper (237,9 млн), Batman Forever (236,5 млн) и Dumb and Dumber (231,3 млн).

Полученные результаты показывают, что фильмы с наибольшими бюджетами не обязательно занимают первые места по разнице между кассовыми сборами и затратами на производство.

При этом данный показатель не является чистой прибылью, поскольку не учитывает маркетинговые расходы, долю кинотеатров и другие затраты.

#### Фильмы с наибольшей продолжительностью `longest(n)`

Рассмотрим продолжительность фильмов в загруженной выборке. Метод `longest(n)` преобразует указанное время в минуты и позволяет определить картины с наибольшей продолжительностью.

In [7]:
%timeit -n 1 -r 1 links.longest(20)
links.longest(20)

318 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


{'Braveheart (1995)': 178,
 'Hoop Dreams (1994)': 174,
 'Strange Days (1995)': 145,
 'Apollo 13 (1995)': 140,
 'Rob Roy (1995)': 139,
 'Burnt by the Sun (Utomlyonnye solntsem) (1994)': 135,
 'Waterworld (1995)': 135,
 'Scarlet Letter, The (1995)': 135,
 'Bridges of Madison County, The (1995)': 135,
 'First Knight (1995)': 134,
 'Dolores Claiborne (1995)': 132,
 'Showgirls (1995)': 131,
 'Disclosure (1994)': 128,
 'Die Hard: With a Vengeance (1995)': 128,
 'Clockers (1995)': 128,
 'Ed Wood (1994)': 127,
 'Eat Drink Man Woman (Yin shi nan nu) (1994)': 123,
 'Immortal Beloved (1994)': 121,
 'Crumb (1994)': 121,
 'Batman Forever (1995)': 121}

Braveheart и Hoop Dreams заметно отрываются от остальных — 178 и 174 минуты против 145 у третьего места. Это эпик и документальный фильм, и их длина выделяет их из общей массы.

Смешаны фильмы очень разного масштаба: от блокбастера Apollo 13 с бюджетом $52M до документального *Crumb* ($3M сборов) и Clockers Спайка Ли. Длительность с бюджетом не связана — короткий фильм может быть дорогим, длинный — дешёвым.

Batman Forever с хронометражем 121 минута замыкает двадцатку, хотя по бюджету он второй в выборке ($100M). Дорогой, но не длинный

#### Фильмы с наибольшими затратами на одну минуту `top_cost_per_minute(n)`

Помимо общего бюджета, интересно сравнить затраты на производство с учётом продолжительности фильмов. Абсолютный бюджет сам по себе мало что говорит: $175M на два с половиной часа и $175M на полтора — разные вещи.

Метод `top_cost_per_minute(n)` рассчитывает отношение бюджета фильма к его продолжительности в минутах. Это позволяет определить фильмы с наибольшими затратами на одну минуту экранного времени.

In [8]:
%timeit -n 1 -r 1 links.top_cost_per_minute(20)
links.top_cost_per_minute(20)

278 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


{'Waterworld (1995)': 1296296.3,
 'Judge Dredd (1995)': 937500.0,
 'Batman Forever (1995)': 826446.28,
 'Die Hard: With a Vengeance (1995)': 703125.0,
 'Under Siege 2: Dark Territory (1995)': 606060.61,
 'Jade (1995)': 526315.79,
 'Up Close and Personal (1996)': 504201.68,
 'Casper (1995)': 500000.0,
 'Congo (1995)': 462962.96,
 'Crimson Tide (1995)': 456896.55,
 'Drop Zone (1994)': 445544.55,
 'Disclosure (1994)': 429687.5,
 'First Knight (1995)': 410447.76,
 'Braveheart (1995)': 404494.38,
 'Apollo 13 (1995)': 371428.57,
 'Scarlet Letter, The (1995)': 370370.37,
 'City Hall (1996)': 360360.36,
 'Showgirls (1995)': 343511.45,
 'Boomerang (1992)': 341880.34,
 'Nine Months (1995)': 339805.83}

Сверху Waterworld — около 1,3 млн долларов на минуту и отрывается от второго места примерно в 1,4 раза. У него и самый большой бюджет в выборке ($175M), и не самый длинный хронометраж (135 мин) — комбинация даёт максимум. Дальше Judge Dredd (937,5 тыс.), Batman Forever (826 тыс.), Die Hard: With a Vengeance (703 тыс.) и Under Siege 2 (606 тыс.). Пятёрка лидеров — почти целиком боевики и фантастика с большими бюджетами, что логично: жанры дорогие, а хронометраж у них средний, так что бюджет на минуту получается высоким.

#### Суммарная продолжительность фильмов по режиссёрам `longest_by_director()`

Рассмотрим, как распределяется общая продолжительность фильмов между режиссёрами в выбранной выборке.

Метод `longest_by_director()` суммирует продолжительность всех фильмов каждого режиссёра и сортирует полученные значения по убыванию. Это позволяет определить, какие режиссёры представлены наибольшим суммарным экранным временем.

In [9]:
%timeit -n 1 -r 1 links.longest_by_director()
links.longest_by_director()

328 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


{'Kevin Smith': 187,
 'Mel Gibson': 178,
 'Steve James': 174,
 'Kathryn Bigelow': 145,
 'Ron Howard': 140,
 'Michael Caton-Jones': 139,
 'Nikita Mikhalkov': 135,
 'Kevin Reynolds': 135,
 'Roland Joffé': 135,
 'Clint Eastwood': 135,
 'Jerry Zucker': 134,
 'Taylor Hackford': 132,
 'Paul Verhoeven': 131,
 'Barry Levinson': 128,
 'John McTiernan': 128,
 'Spike Lee': 128,
 'Tim Burton': 127,
 'Ang Lee': 123,
 'Bernard Rose': 121,
 'Terry Zwigoff': 121,
 'Joel Schumacher': 121,
 'Todd Haynes': 119,
 'Michael Bay': 119,
 'Mike Nichols': 119,
 'Jon Avnet': 119,
 'Reginald Hudlin': 117,
 'Jon Blair': 117,
 'Tony Scott': 116,
 'John Dahl': 116,
 'Herbert Ross': 115,
 'Martha Coolidge': 115,
 'Garry Marshall': 114,
 'Irwin Winkler': 114,
 'Martin Scorsese': 114,
 'Milcho Manchevski': 113,
 'Mike Newell': 113,
 'Andrew Davis': 113,
 'Wayne Wang': 112,
 'Gérard Corbiau': 111,
 'Lawrence Kasdan': 111,
 'Agnieszka Holland': 111,
 'Harold Becker': 111,
 'Randall Miller': 109,
 'Peter Jackson': 109,
 '

Наибольшая суммарная продолжительность фильмов наблюдается у  Kevin Smith — 187 минут. Дальше Mel Gibson (178), Steve James (174), Kathryn Bigelow (145) и Ron Howard (140).

Тут важно понимать, что именно показывает цифра. У Kevin Smith в выборке два фильма — Clerks (92 мин) и Mallrats (95 мин). В сумме 187. У Mel Gibson и Steve James — по одному фильму (Braveheart, 178, и Hoop Dreams, 174), и они всё равно почти догнали лидера. То есть первое место Kevin Smith — это не «самый длинный», а «больше всего суммарного времени за счёт количества картин».

#### Выводы по Links

В рамках работы над классом `Links` была реализована загрузка дополнительной информации о фильмах с платформы TMDB.

Для анализа взяли диапазон ID от 100 до 250.Подтянули названия, режиссёров, бюджеты, мировые сборы и продолжительность.

Что показал анализ:

- **Бюджеты**. Waterworld — $175M, что в 1,75 раза больше второго места (*Batman Forever*, $100M). Дальше Judge Dredd и Die Hard: With a Vengeance (по $90M), *Braveheart* ($72M). В топе много боевиков и фантастики, но есть и неожиданности вроде мелодрамы Up Close & Personal ($60M).
- **Разница сборов и бюджета**. Здесь порядок другой: Apollo 13 ($303,2M), *Die Hard: With a Vengeance* ($276,1M), Casper ($237,9M), *Batman Forever* ($236,5M), Dumb and Dumber ($231,3M). *Waterworld* с максимальным бюджетом — только 15-й ($89,2M). Дорого снять не значит много заработать.
- **Продолжительность**. Braveheart — 178 минут, дальше Hoop Dreams (174), Strange Days (145), Apollo 13 (140), Rob Roy (139). Вся двадцатка укладывается в 121–178 минут. С бюджетом длина не связана: рядом стоят блокбастер Apollo 13 с бюджетом $52M и документальный Crumb.
- **Бюджет на минуту**. Снова Waterworld (~1,3 млн), дальше Judge Dredd (937,5 тыс.), Batman Forever (826 тыс.), Die Hard: With a Vengeance (703 тыс.), Under Siege 2 (606 тыс.). Braveheart с бюджетом $72M и хронометражем 178 минут опускается на 14-е место — длина работает против него.
- **Режиссёры**. Чаще всех встречается Kevin Smith — два фильма (Clerks, Mallrats), суммарно 187 минут. Но первое место здесь за счёт количества, а не длины: по среднему хронометражу Braveheart Мэла Гибсона (178 минут, один фильм) вышел бы вперёд.

---

### <a id='part4'></a>Часть 4. Семантический анализ (класс Tags)

> В предыдущей части мы посмотрели на фильмы как на коммерческий продукт: изучили их бюджеты, кассовые сборы и хронометраж. Мы выяснили, что дорого снять — не значит много заработать, а длинный фильм не обязательно дорогой.
> 
> Однако, кино — это не только цифры в бухгалтерских отчетах и минуты экранного времени. Это в первую очередь эмоции и смыслы. Для того чтобы понять, *о чем* на самом деле фильм и *что* в нем ищут зрители, мы перейдем от **количественных** к **качественным** данным. 
> 
> В этой части мы проанализируем **неструктурированные текстовые данные — пользовательские теги**. С точки зрения Data Science, это переход от табличных метрик к области *Natural Language Processing (NLP)*, где мы попытаемся извлечь семантику из того, как люди описывают кино.

#### Активность тегов: Что пишут чаще всего?

> Начнем с базового анализа. Метод `most_popular` подсчитывает, какие теги пользователи используют чаще всего. Это покажет нам основные темы, доминирующие в сознании аудитории.

In [ ]:
%timeit -n 1 -r 1
popular_tags = tags.most_popular(15)

print(f"{'Тег':<30} | {'Кол-во использований':<20}")
print("-" * 55)
for tag, count in popular_tags.items():
    print(f"{tag:<30} | {count:<20}")

Тег                            | Кол-во использований
-------------------------------------------------------
In Netflix queue               | 131                 
atmospheric                    | 36                  
superhero                      | 24                  
thought-provoking              | 24                  
Disney                         | 23                  
funny                          | 23                  
surreal                        | 23                  
religion                       | 22                  
dark comedy                    | 21                  
psychology                     | 21                  
quirky                         | 21                  
sci-fi                         | 21                  
suspense                       | 20                  
crime                          | 19                  
twist ending                   | 19                  


#### Анализ: От жанров к эмоциям и атмосфере

Взгляните на этот топ. В отличие от предыдущих анализов, здесь почти нет сухих жанровых категорий. Абсолютный лидер — тег `In Netflix queue` (131 использование), что указывает на сильное *поведенческое смещение* в данных: пользователи часто используют теги не для описания фильма, а как личные заметки (например, «добавить в список»).

Если отбросить этот артефакт, мы увидим настоящую семантику. Пользователи описывают не *жанры* (`sci-fi` или `comedy` здесь на нижних строчках), а *атмосферу* (`atmospheric`, `surreal`, `quirky`), *эмоции* (`funny`, `suspense`) и *сложность сюжета* (`thought-provoking`, `twist ending`).

С точки зрения NLP и Data Science:

1. **Высокая информационная ценность:** Теги вроде `thought-provoking` или `surreal` гораздо ценнее для алгоритмов, чем просто `Action`. Они позволяют отличить глубокий философский `sci-fi` от глупого боевика.

2. **Проблема шума:** Тег `In Netflix queue` — это классический шум, который не имеет отношения к контенту. Если мы подадим эти данные в модель без предварительной очистки, алгоритм решит, что все эти фильмы объединены тем, что когда-то лежали в очереди Netflix.

3. **Рекомендации на основе vibe (настроения):** Такие теги идеально подходят для систем рекомендаций на основе настроения. Пользователи ищут не просто *«комедию»*, а что-то *«атмосферное»* или *«с неожиданной концовкой»*.

#### Длина тегов: От коротких меток к развернутым описаниям.

> Не все теги одинаковы. Одни пользователи ограничиваются одним словом, другие пишут целые предложения. Давайте посмотрим на самые длинные теги (по количеству символов) и на те, в которых больше всего слов. Это покажет, насколько глубоко зрители готовы описывать свои впечатления, и как выглядит "длинный хвост" текстовых данных.

In [3]:
%timeit -n 1 -r 1
longest_tags = tags.longest(5)
most_words_tags = tags.most_words(5)

print("Самые длинные теги (по количеству символов):")
for tag in longest_tags:
    print(f"  - {tag}")

print("\nТеги с наибольшим количеством слов:")
for tag, count in most_words_tags.items():
    print(f"  - {tag} (слов: {count})")

Самые длинные теги (по количеству символов):
  - Something for everyone in this one... saw it without and plan on seeing it with kids!
  - the catholic church is the most corrupt organization in history
  - villain nonexistent or not needed for good story
  - r:disturbing violent content including rape
  - 06 Oscar Nominated Best Movie - Animation

Теги с наибольшим количеством слов:
  - Something for everyone in this one... saw it without and plan on seeing it with kids! (слов: 16)
  - the catholic church is the most corrupt organization in history (слов: 10)
  - villain nonexistent or not needed for good story (слов: 8)
  - 06 Oscar Nominated Best Movie - Animation (слов: 7)
  - It was melodramatic and kind of dumb (слов: 7)


#### Анализ: Неструктурированные данные и высокоразмерные признаки

Этот результат наглядно демонстрирует главную проблему обработки текстов — **отсутствие структуры**. Пользователи используют поле для тегов не по назначению: вместо коротких жанровых меток мы видим целые предложения, личные отзывы, упоминания режиссеров и даже социальные комментарии. С точки зрения машинного обучения это создает несколько трудностей:

1. **Высокая энтропия смыслов:** Разброс смыслов здесь максимален. Один тег описывает сюжет, другой — моральную оценку, третий — наличие сцен насилия. Для алгоритма это огромный объем **информационного шума**, который не поддается категоризации.

2. **Разреженность данных:** Почти все эти длинные теги уникальны. Если мы построим матрицу «Фильм-Слово», она будет состоять из нулей.

3. **Необходимость NLP-пайплайна:** Чтобы алгоритм смог это обработать, потребуется сложная предобработка: **токенизация** (разбивка на слова), **удаление стоп-слов** (предлогов, союзов), **лемматизация** (приведение слов к словарной форме) и **векторизация** (TF-IDF или эмбеддинги).

Без этой очистки алгоритм машинного обучения не сможет понять, что `saw it without kids` (смотрел без детей) — это не описание жанра, а личный контекст просмотра, который бесполезен для рекомендаций другим пользователям.

#### Пересечение: Длинные и глубокие теги

> Мы увидели, что в базе есть как короткие ярлыки, так и целые предложения. Но какие из них действительно важны? Чтобы отфильтровать случайный шум и найти самые информативные описания, давайте посмотрим на пересечение: теги, которые одновременно являются самыми длинными по количеству символов и содержат больше всего слов (метод `most_words_and_longest`).

In [4]:
%timeit -n 1 -r 1
intersection = tags.most_words_and_longest(30)
print("Самые длинные уникальные теги:")
for tag in intersection:
    print(f"  - {tag}")

Самые длинные уникальные теги:
  - villain nonexistent or not needed for good story
  - Everything you want is here
  - stop using useless characters for filler
  - Academy award (Best Supporting Actress)
  - the catholic church is the most corrupt organization in history
  - coulda been a contender
  - It was melodramatic and kind of dumb
  - Something for everyone in this one... saw it without and plan on seeing it with kids!
  - start of a beautiful friendship
  - r:disturbing violent content including rape
  - Oscar (Best Music - Original Score)
  - Oscar (Best Effects - Visual Effects)
  - 06 Oscar Nominated Best Movie - Animation
  - stupid is as stupid does


#### Анализ: Семантическая неоднородность текста

Мы видим наглядный пример того, насколько неоднородными могут быть текстовые данные, генерируемые пользователями. Метод пересечения отобрал самые длинные и многозначные фразы, и мы видим, что они относятся к совершенно разным категориям:

1. **Метаданные и факты:** Упоминания наград `Academy award, 06 Oscar Nominated` или возрастных рейтингов `r:disturbing violent content`. Это структурированная информация, которую пользователи просто скопировали из других источников.

2. **Субъективные мнения:** Личные оценки вроде `It was melodramatic and kind of dumb` или `villain nonexistent`.

3. **Контекст просмотра:** `Something for everyone in this one...` — это уже не описание фильма, а личный опыт.

Из-за такого смешения смыслов использовать эти данные для обучения модели «как есть» не получится: алгоритм не найдет логики в том, что общего между *Оскаром* и *сценой насилия*.

Поэтому на следующем этапе обработки текста потребуется **снижение размерности** и **кластеризация**. Алгоритм (например, K-Means или тематическое моделирование LDA) должен автоматически сгруппировать эти теги: отдельно — блок про награды, отдельно — про цензуру, отдельно — про эмоциональные отзывы. Только после такой фильтрации и структуризации текст можно будет эффективно использовать в рекомендательной системе.

#### Поиск по контексту

> Теперь проверим, как работают базовые алгоритмы текстового поиска. Метод `tags_with` позволяет найти все уникальные теги, содержащие определенную подстроку (регистронезависимо). Для примера посмотрим, какими словами пользователи описывают научную фантастику (подстрока `sci-fi`).

In [5]:
%timeit -n 1 -r 1
sci_fi_tags = tags.tags_with("sci-fi")
print(f"Найдено уникальных тегов с 'sci-fi': {len(sci_fi_tags)}")
print("Примеры найденных тегов:")
for tag in sci_fi_tags[:15]: 
    print(f"  - {tag}")

Найдено уникальных тегов с 'sci-fi': 5
Примеры найденных тегов:
  - Sci-Fi
  - Sci-fi
  - classic sci-fi
  - intelligent sci-fi
  - sci-fi


#### Анализ: Проблема неструктурированности и регистра

Этот небольшой пример отлично демонстрирует главную проблему работы с текстовыми данными — **отсутствие стандартизации**. Метод tags_with нашел 5 уникальных тегов, содержащих `sci-fi`, но если присмотреться, среди них есть `Sci-Fi` (с заглавными буквами) и `sci-fi` (со строчными). Для компьютера это два абсолютно разных слова, хотя для человека смысл очевиден.

Кроме того, мы видим контекстные дополнения: `classic sci-fi` и `intelligent sci-fi`.

С точки зрения подготовки данных, это означает, что перед подачей текста в модель машинного обучения потребуется нормализация. В частности, нужно будет:

1. Привести все теги к нижнему регистру.

2. Использовать *стемминг или лемматизацию*, чтобы объединить похожие формы слов.

Только после такой очистки алгоритм сможет правильно посчитать частоту: сейчас он думает, что это 5 разных тегов, хотя на самом деле это один популярный жанр.

#### Уникальность текстов: Проблема «Длинного хвоста»

> В анализе текста важно понимать, насколько разнообразен словарь тегов. Я добавил в класс `Tags` новый метод `unique_tags_count()`, который считает, сколько тегов встречаются в базе ровно один раз. Это покажет уровень **разреженности** текстовых данных.

In [6]:
%timeit -n 1 -r 1
unique_count = tags.unique_tags_count()
total_unique = len(tags._get_unique_tags())

print(f"Всего уникальных тегов в базе: {total_unique}")
print(f"Из них встречаются только 1 раз (синглтоны): {unique_count}")
print(f"Доля синглтонов: {round(unique_count / total_unique * 100, 1)}%")

Всего уникальных тегов в базе: 1589
Из них встречаются только 1 раз (синглтоны): 1038
Доля синглтонов: 65.3%


#### Анализ: Проклятие размерности

Полученный результат — 65.3% — очень показателен. Он означает, что почти две трети всех тегов в базе (1038 из 1589) встречаются ровно один раз. Это классическая иллюстрация проблемы *«длинного хвоста»*, которая характерна для любых текстовых данных, создаваемых пользователями.

С точки зрения машинного обучения, такая структура создает серьезную проблему. Если мы попытаемся построить матрицу признаков «Фильм-Тег» (где столбцы — это все 1589 уникальных слов), она окажется колоссальной, но при этом крайне разреженной. Почти во всех ячейках будут нули. Алгоритму будет крайне сложно найти закономерности в таких данных.

Чтобы решить эту проблему и заставить текст работать на рекомендательную систему, применяют специальную предобработку:

1. **Стемминг и лемматизацию** — приведение разных форм одного слова к базовой основе (чтобы `sci-fi` и `Sci-Fi` стали одним признаком).

2. **Фильтрацию по частоте** — удаление редких слов (синглтонов), которые являются шумом, и слишком частых стоп-слов.

3. **Векторные эмбеддинги (Word2Vec, BERT)** — превращение слов в числовые векторы, где близкие по смыслу слова находятся рядом, что позволяет модели обобщать данные даже на редких тегах.

#### Итоговые выводы по части 4 (Класс Tags)

Анализ пользовательских тегов показал, что данные MovieLens — это не просто матрица оценок, а богатое семантическое пространство. Мы выяснили, что:

1. **Текст — это неструктурированные данные:** Пользователи используют поле для тегов не по прямому назначению (как жанровые ярлыки), а как личные заметки. Мы встретили там имена режиссеров, возрастные рейтинги, целые рецензии и даже тег `In Netflix queue`.

2. **Высокая информационная ценность:** Несмотря на шум, теги вроде `atmospheric`, `surreal` или `thought-provoking` несут гораздо больше смысла для алгоритма, чем базовые жанры. Они описывают эмоции и атмосферу, что позволяет строить рекомендации на основе настроения.

3. **Проблема «Длинного хвоста» и разреженность:** Мы обнаружили, что более 65% тегов встречаются в базе всего один раз (синглтоны). Это создает огромную разреженную матрицу, с которой стандартным алгоритмам машинного обучения сложно работать.

4. **Потребность в NLP-пайплайне:** Чтобы использовать эти данные для рекомендаций, потребуется серьезная предобработка текста: приведение к нижнему регистру, лемматизация, удаление шума и векторизация (например, через TF-IDF или эмбеддинги).

**Общий итог:** Текстовые данные в MovieLens — это одновременно и ценнейший ресурс, и серьезный вызов. Сырые теги несут в себе много шума и нестандартизированной информации, но после применения методов обработки естественного языка (NLP) они могут стать мощным инструментом для семантического поиска и построения более умных рекомендательных систем.

---

## <a id='conclusion'></a>Общий вывод по проекту MovieLens

В ходе данного исследования мы провели комплексный разведочный анализ данных (EDA) датасета MovieLens, охватив четыре ключевых измерения: сам контент (`Movies`), коммерческие показатели (`Links`), поведение пользователей (`Ratings`) и семантику (`Tags`). 

Несмотря на то, что каждый класс решал свою локальную задачу, вместе они позволяют сделать несколько фундаментальных выводов о том, как устроены данные для рекомендательных систем.

**Что мы выяснили:**

1. **Контент (Movies):** В базе доминируют драмы и комедии, а основная масса фильмов приходится на 90-е и 2000-е годы. Жанровая разметка часто избыточна (до 10 жанров у одного фильма), что размывает категоризацию и усложняет поиск.
2. **Коммерция (Links):** Анализ данных с TMDB показал, что огромные бюджеты не гарантируют высокой прибыли. Самым успешным по разнице сборов и бюджета стал *Apollo 13*. Также мы выяснили, что хронометраж фильма никак не коррелирует с его бюджетом.
3. **Поведение зрителей (Ratings):** Мы обнаружили серьезные **смещения данных**. Оценки имеют положительный сдвиг (люди хвалят кино), аудитория крайне неоднородна (20 сверхактивных пользователей сгенерировали четверть всех оценок), а разброс средних баллов у зрителей огромен. Чтобы найти объективные шедевры, мы применили **байесовское сглаживание** и **пороговую фильтрацию**, отсеяв статистический шум.
4. **Семантика (Tags):** Анализ текстовых меток подтвердил проблему *«длинного хвоста»*. Более 65% тегов — это уникальные слова (синглтоны). Они несут высокую эмоциональную ценность (атмосфера, настроение), но создают разреженные матрицы, требующие сложной NLP-обработки (лемматизация, векторизация).

**Ввывод:**
Данный проект наглядно демонстрирует, что сырые данные в рекомендательных системах практически бесполезны. Будь то оценки, финансовая статистика или текстовые теги — всё это полно шума, выбросов и смещений. 

Построение качественной рекомендательной системы требует не просто пересчета средних значений, а применения продвинутых статистических методов (сглаживание, нормализация весов, фильтрация по уровню доверия) и алгоритмов обработки текста. Только очистив данные от статистического мусора, можно выявить истинные предпочтения зрителей и построить работающий алгоритм, который будет предлагать пользователям именно то, что им понравится.